In [115]:
# ───────────────────────────────────────────────────────────────
# 0. 라이브러리 & 데이터 로드
# ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

# 역 피처셋
df = pd.read_csv("./data/processed/feature_engineering_final.csv", encoding="utf-8-sig")

# 지반침하 사고 지점
acc = pd.read_csv("./data/processed/지반침하_고위험지역_주소.csv", encoding="utf-8-sig")

# 좌표 보강 (lat1/lon1가 비어 있으면 lat2/lon2 사용)
acc["lat"] = acc["lat1"].fillna(acc["lat2"])
acc["lon"] = acc["lon1"].fillna(acc["lon2"])
acc = acc.dropna(subset=["lat", "lon"]).reset_index(drop=True)

print(f"역 개수  : {len(df)}")
print(f"사고 지점: {len(acc)}")

역 개수  : 239
사고 지점: 54


In [116]:
# ───────────────────────────────────────────────────────────────
# 1. BallTree 구축 (위·경도를 라디안으로 변환)
# ───────────────────────────────────────────────────────────────
R_EARTH_KM = 6371.0088                    # 평균 지구 반경(km)

acc_coords  = np.deg2rad(acc[["lat", "lon"]].values)
stn_coords  = np.deg2rad(df[["위도", "경도"]].values)

tree = BallTree(acc_coords, metric="haversine")

# 검색 반경 제한 없이 가장 가까운 사고 지점 1개 반환
dist_radians, _ = tree.query(stn_coords, k=1)
dist_meters = dist_radians.flatten() * R_EARTH_KM * 1000   # km → m

df["dist_to_sinkhole_m"] = dist_meters.round(1)

In [117]:
# ───────────────────────────────────────────────────────────────
# 2. 거리 기반 라벨링
#    예시: 0 = 안전 (>300 m)
#          1 = 주의 (150–300 m)
#          2 = 위험 (≤150 m)
# ───────────────────────────────────────────────────────────────
def label_by_dist(d):
    if d <= 300:
        return 2
    elif d <= 500:
        return 1
    else:
        return 0

df["sinkhole_risk_label"] = df["dist_to_sinkhole_m"].apply(label_by_dist)

df[["station_name", "dist_to_sinkhole_m", "sinkhole_risk_label"]].head()

,station_name,dist_to_sinkhole_m,sinkhole_risk_label
0,가락시장,8724.7,0
1,가산디지털단지,649.3,0
2,강남,3583.4,0
3,강남구청,1674.6,0
4,강동,8425.8,0


In [118]:
df['sinkhole_risk_label'].unique()

array([0, 1, 2])

In [119]:
df['sinkhole_risk_label'].value_counts()

sinkhole_risk_label
0    229
1      8
2      2
Name: count, dtype: int64

In [120]:
df[df['sinkhole_risk_label'] == 2]

,station_name,연번,호선,층수,형식,지반고,레일면고,위도,경도,비고,...,road_construction_count_500m,comprehensive_risk_score,is_island_platform,is_side_platform,is_transfer_station,line_1to4,line_5to8,risk_level,dist_to_sinkhole_m,sinkhole_risk_label
154,안국,71,3,B4,섬식,133.10,113.19,37.576562,126.98547,NaN,...,533.0,80.758864,1,0,0,1,0,Low,268.9,2
203,종로5가,5,1,B2,상대식,121.75,109.26,37.570971,127.00190,NaN,...,533.0,81.221602,0,1,0,1,0,Low,265.8,2


In [121]:
df[df['sinkhole_risk_label'] == 1]

,station_name,연번,호선,층수,형식,지반고,레일면고,위도,경도,비고,...,road_construction_count_500m,comprehensive_risk_score,is_island_platform,is_side_platform,is_transfer_station,line_1to4,line_5to8,risk_level,dist_to_sinkhole_m,sinkhole_risk_label
54,대흥,193,6,B3,섬식,114.97,94.10,37.547732,126.942214,NaN,...,533.0,80.810232,1,0,0,0,1,Low,364.6,1
112,상왕십리,17,2,B2,상대식,134.38,119.21,37.564504,127.028872,NaN,...,533.0,80.641970,0,1,0,1,0,Low,399.9,1
157,압구정,79,3,B2,상대식,121.58,109.19,37.526169,127.028502,NaN,...,533.0,80.366488,0,1,0,1,0,Low,498.0,1
176,왕십리,18,2,B2,상대식,118.41,104.86,37.561159,127.035505,"5호선,분당선,경의중앙선환승",...,533.0,83.726402,0,1,1,1,0,Medium,311.6,1
201,종각,3,1,B2,상대식,128.77,116.24,37.570203,126.983116,NaN,...,533.0,84.556580,0,1,0,1,0,High,333.2,1
202,종로3가,4,1,B2,상대식,124.38,112.04,37.570429,126.992095,"3,5호선환승",...,533.0,81.467162,0,1,1,1,0,Low,345.1,1
232,혜화,107,4,B2,상대식,128.26,114.28,37.582116,127.001759,NaN,...,533.0,81.542628,0,1,0,1,0,Low,443.2,1
235,화곡,129,5,B3,섬식,125.12,105.12,37.541585,126.840436,NaN,...,533.0,81.604200,1,0,0,0,1,Low,328.0,1
